# Does late delivery *cause* worse reviews? — Olist Marketplace

Earlier SQL analysis showed late orders get far worse reviews, but that is a **correlation**: maybe late orders are also pricier, heavier, or further away. This notebook tests whether the effect holds up once we **control for confounders**, and builds a model of what drives a bad review.

**Steps:** (1) a significance test on the raw gap, (2) a logistic regression controlling for price, freight, item count, and estimated delivery time, (3) a Random Forest with feature importance.

*Data: Olist Brazilian E-Commerce Public Dataset (raw CSVs in `../data/`).*

In [1]:
import pandas as pd, numpy as np

orders = pd.read_csv('../data/olist_orders_dataset.csv')
items  = pd.read_csv('../data/olist_order_items_dataset.csv')
prods  = pd.read_csv('../data/olist_products_dataset.csv')
cust   = pd.read_csv('../data/olist_customers_dataset.csv')
trans  = pd.read_csv('../data/product_category_name_translation.csv')
revs   = pd.read_csv('../data/olist_order_reviews_dataset.csv')

# delivered orders, with delivery timing
o = orders[orders['order_status']=='delivered'].copy()
for c in ['order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date']:
    o[c] = pd.to_datetime(o[c], errors='coerce')
o['is_late']  = (o['order_delivered_customer_date'] > o['order_estimated_delivery_date']).astype(int)
o['est_days'] = (o['order_estimated_delivery_date'] - o['order_purchase_timestamp']).dt.days

# order-level features
agg = items.groupby('order_id').agg(price=('price','sum'),
                                    freight=('freight_value','sum'),
                                    n_items=('order_item_id','count')).reset_index()
revs['a'] = pd.to_datetime(revs['review_answer_timestamp'], errors='coerce')
r = revs.sort_values('a').drop_duplicates('order_id', keep='last')[['order_id','review_score']]

df = (o.merge(agg, on='order_id', how='left')
        .merge(cust[['customer_id','customer_state']], on='customer_id', how='left')
        .merge(r, on='order_id', how='left'))
df = df.dropna(subset=['review_score','price','est_days']).copy()
df['is_low']     = (df['review_score'] <= 2).astype(int)   # 1-2 star review
df['log_price']  = np.log1p(df['price'])
df['log_freight']= np.log1p(df['freight'])
print(f"Orders analyzed: {len(df):,}")
df[['is_late','is_low','price','freight','n_items','est_days']].describe().round(2)

Orders analyzed: 95,832


,is_late,is_low,price,freight,n_items,est_days
count,95832.00,95832.00,95832.00,95832.00,95832.00,95832.00
mean,0.08,0.13,136.80,22.76,1.14,23.37
std,0.27,0.33,207.78,21.52,0.53,8.76
min,0.00,0.00,0.85,0.00,1.00,2.00
25%,0.00,0.00,45.90,13.84,1.00,18.00
50%,0.00,0.00,86.25,17.16,1.00,23.00
75%,0.00,0.00,149.90,23.99,1.00,28.00
max,1.00,1.00,13440.00,1794.96,21.00,155.00


## 1. Significance test — is the raw gap real?

Two-proportion z-test on the low-review rate (share of 1–2 star reviews) for late vs on-time orders.

In [2]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

late, ont = df[df.is_late==1], df[df.is_late==0]
counts = [late.is_low.sum(), ont.is_low.sum()]
nobs   = [len(late), len(ont)]
z, p   = proportions_ztest(counts, nobs)
lo, hi = confint_proportions_2indep(counts[0], nobs[0], counts[1], nobs[1])

print(f"low-review rate  |  late = {counts[0]/nobs[0]:.1%}   on-time = {counts[1]/nobs[1]:.1%}")
print(f"difference       = {counts[0]/nobs[0]-counts[1]/nobs[1]:.1%}   95% CI [{lo:.1%}, {hi:.1%}]")
print(f"z = {z:.1f},  p = {p:.1e}")

low-review rate  |  late = 54.1%   on-time = 9.2%
difference       = 44.8%   95% CI [43.7%, 46.0%]
z = 112.7,  p = 0.0e+00


The gap is huge and highly significant, but this does not yet prove *cause*. Late orders might differ in other ways. Next we control for those.

## 2. Logistic regression — does it survive controls?

We model the probability of a 1–2 star review from **is_late** plus confounders (price, freight, item count, estimated delivery days). If the effect were just those confounders, the `is_late` coefficient would shrink toward zero.

In [3]:
import statsmodels.formula.api as smf

model = smf.logit('is_low ~ is_late + log_price + log_freight + n_items + est_days', data=df).fit(disp=0)
odds  = np.exp(model.params['is_late'])
ci    = np.exp(model.conf_int().loc['is_late'])
print(f"Odds ratio for LATE delivery = {odds:.1f}   (95% CI {ci[0]:.1f}-{ci[1]:.1f}),  p = {model.pvalues['is_late']:.1e}")
print(f"-> Holding price, freight, item count and estimated delivery time constant,")
print(f"   a late order has ~{odds:.0f}x the odds of a 1-2 star review.")
model.summary()

Odds ratio for LATE delivery = 13.0   (95% CI 12.4-13.7),  p = 0.0e+00
-> Holding price, freight, item count and estimated delivery time constant,
   a late order has ~13x the odds of a 1-2 star review.


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                 is_low   No. Observations:                95832
Model:                          Logit   Df Residuals:                    95826
Method:                           MLE   Df Model:                            5
Date:                Wed, 08 Jul 2026   Pseudo R-squ.:                  0.1412
Time:                        13:35:29   Log-Likelihood:                -31495.
converged:                       True   LL-Null:                       -36675.
Covariance Type:            nonrobust   LLR p-value:                     0.000
===============================================================================
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      -3.8772      0.065    -59.915      0.000      -4.004      -3.750
is_late         2.5686      0.027     96.630      0.000       2.516       2.621
log_price       0.0316      0.013      2.432      0.015       0.006       0.057
log_freight     0.1016      0.024      4.159      0.000       0.054       0.149
n_items         0.5267      0.018     29.711      0.000       0.492       0.561
est_days        0.0205      0.001     16.462      0.000       0.018       0.023
===============================================================================
"""

**The effect survives.** Controlling for the obvious confounders, late delivery still multiplies the odds of a bad review about 13x. This is strong, defensible evidence that the delivery experience itself, not just correlated factors, drives dissatisfaction.

## 3. What drives a bad review? — Random Forest

A predictive model to rank the drivers and sanity-check the story.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

feats = ['is_late','log_price','log_freight','n_items','est_days']
X, y  = df[feats], df['is_low']
Xtr,Xte,ytr,yte = train_test_split(X, y, test_size=.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42,
                            n_jobs=-1, class_weight='balanced').fit(Xtr, ytr)
auc = roc_auc_score(yte, rf.predict_proba(Xte)[:,1])
print(f"Test AUC = {auc:.3f}")
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print("\nFeature importance:"); print(imp.round(3))

Test AUC = 0.720

Feature importance:
is_late        0.592
n_items        0.139
log_freight    0.114
log_price      0.083
est_days       0.073
dtype: float64


## Conclusion & recommendation

- Late delivery is not just correlated with bad reviews, it **drives** them: after controlling for price, freight, item count, and estimated delivery time, a late order has roughly **13x the odds** of a 1–2 star review, and the model ranks it the **top driver** by a wide margin.
- Combined with the fact that about **97% of customers never buy again**, the first delivery largely decides customer value.
- **Recommendation:** test a more conservative delivery promise (under-promise, over-deliver) to cut the late-vs-estimate rate. The experiment design is in `EXPERIMENT_DESIGN.md`; this analysis quantifies the size of the prize and confirms the lever is real.